# Bloomington Mortality-Weather Regression

Replicates the same methodology used for Austin's mortality-weather regression (moon-phase-weather-shelter-analysis), applied independently to Bloomington's real shelter and weather data, to test whether the temperature-mortality relationship found at Austin generalizes to a second, independent shelter.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

df = pd.read_csv('/kaggle/input/datasets/micahluftig/bloomington-model-for-eval/bloomington_model_ready.csv')
weather = pd.read_csv('/kaggle/input/datasets/micahluftig/bloomington-clean-weather/bloomington_clean_weather.csv')
weather['datetime'] = pd.to_datetime(weather['datetime'])
df['deceaseddate'] = pd.to_datetime(df['deceaseddate'], errors='coerce')

## Verifying the Data Coverage Window

Before fitting anything, check real record density by year -- an earlier version of this project incorrectly used the full date range and mistook years with essentially zero real shelter records for genuine "no deaths" days.

In [2]:
print(df['datetime_in'].astype(str).str[:4].value_counts().sort_index())
# Real, usable density only starts in 2017 -- everything before is a handful of scattered
# records, not real "no incidents" days. Restricting the regression to 2017-01-01 through
# 2019-08-30 avoids treating missing data as zero-mortality days.

datetime_in
2009       2
2012       1
2013       2
2015       2
2016       3
2017    2507
2018    2886
2019    1885
Name: count, dtype: int64


## Identifying True Deaths

Using `outcome_category == 'deceased'` and `deceaseddate` -- the same real, verified fields used elsewhere in this project (not a raw event count, which was the source of a real data bug caught and corrected in the original Austin moon-phase analysis).

In [3]:
deaths = df[df['outcome_category'] == 'deceased'].copy()
deaths['death_date_only'] = deaths['deceaseddate'].dt.normalize()

daily_deaths = deaths.groupby('death_date_only').size().reset_index(name='death_count')

calendar = pd.DataFrame({'death_date_only': pd.date_range('2017-01-01', '2019-08-30', freq='D')})
daily_full = calendar.merge(daily_deaths, on='death_date_only', how='left')
daily_full['death_count'] = daily_full['death_count'].fillna(0)

model_df = daily_full.merge(weather, left_on='death_date_only', right_on='datetime', how='left')
model_df = model_df.dropna(subset=['temp', 'sealevelpressure', 'precip'])

print(f"Days in window: {len(model_df)}")
print(f"Mean daily deaths: {model_df['death_count'].mean():.4f}")
print(f"Variance/Mean ratio: {model_df['death_count'].var()/model_df['death_count'].mean():.4f}")

Days in window: 947
Mean daily deaths: 0.2629
Variance/Mean ratio: 1.1399


## Negative Binomial Regression

Same standardized-predictor approach as Austin: fit temperature alone, then check whether pressure or precipitation remain significant once temperature is controlled for.

In [4]:
model_df['temp_z'] = (model_df['temp'] - model_df['temp'].mean()) / model_df['temp'].std()
model_df['pressure_z'] = (model_df['sealevelpressure'] - model_df['sealevelpressure'].mean()) / model_df['sealevelpressure'].std()
model_df['precip_z'] = (model_df['precip'] - model_df['precip'].mean()) / model_df['precip'].std()
y = model_df['death_count']

X_temp = sm.add_constant(model_df[['temp_z']])
nb_temp = sm.NegativeBinomial(y, X_temp).fit(method='bfgs', maxiter=200)
print(nb_temp.summary())

Optimization terminated successfully.
         Current function value: 0.629629
         Iterations: 14
         Function evaluations: 16
         Gradient evaluations: 16
                     NegativeBinomial Regression Results                      
Dep. Variable:            death_count   No. Observations:                  947
Model:               NegativeBinomial   Df Residuals:                      945
Method:                           MLE   Df Model:                            1
Date:                Mon, 10 Aug 2026   Pseudo R-squ.:                 0.02264
Time:                        22:54:24   Log-Likelihood:                -596.26
converged:                       True   LL-Null:                       -610.07
Covariance Type:            nonrobust   LLR p-value:                 1.477e-07
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -1.4027      0.070    -1

In [5]:
# Controlling for temperature, is pressure still significant?
X_tp = sm.add_constant(model_df[['temp_z', 'pressure_z']])
nb_tp = sm.NegativeBinomial(y, X_tp).fit(method='bfgs', maxiter=200)
print(nb_tp.summary())

Optimization terminated successfully.
         Current function value: 0.629133
         Iterations: 16
         Function evaluations: 18
         Gradient evaluations: 18
                     NegativeBinomial Regression Results                      
Dep. Variable:            death_count   No. Observations:                  947
Model:               NegativeBinomial   Df Residuals:                      944
Method:                           MLE   Df Model:                            2
Date:                Mon, 10 Aug 2026   Pseudo R-squ.:                 0.02341
Time:                        22:54:24   Log-Likelihood:                -595.79
converged:                       True   LL-Null:                       -610.07
Covariance Type:            nonrobust   LLR p-value:                 6.288e-07
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -1.4039      0.070    -1

In [6]:
# Controlling for temperature, is precipitation still significant?
X_tprecip = sm.add_constant(model_df[['temp_z', 'precip_z']])
nb_tprecip = sm.NegativeBinomial(y, X_tprecip).fit(method='bfgs', maxiter=200)
print(nb_tprecip.summary())

Optimization terminated successfully.
         Current function value: 0.629496
         Iterations: 14
         Function evaluations: 16
         Gradient evaluations: 16
                     NegativeBinomial Regression Results                      
Dep. Variable:            death_count   No. Observations:                  947
Model:               NegativeBinomial   Df Residuals:                      944
Method:                           MLE   Df Model:                            2
Date:                Mon, 10 Aug 2026   Pseudo R-squ.:                 0.02284
Time:                        22:54:24   Log-Likelihood:                -596.13
converged:                       True   LL-Null:                       -610.07
Covariance Type:            nonrobust   LLR p-value:                 8.869e-07
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -1.4035      0.071    -1

## Result

Temperature remains highly significant (p < 0.001) after controlling for either pressure or precipitation; neither pressure (p = 0.33) nor precipitation (p = 0.61) is significant once temperature is included. This independently replicates the same pressure-is-a-temperature-proxy finding from the original Austin analysis, at a second, independent shelter, in a different climate, using a dataset roughly 1/7th the density of Austin's.

In [7]:
# Export the fitted parameters this app's Monte Carlo simulation actually uses
temp_mean_f = model_df['temp'].mean() * 9/5 + 32
temp_std_f = model_df['temp'].std() * 9/5
pct_change_per_degF = np.exp(nb_temp.params['temp_z'] / temp_std_f) - 1

bloomington_mortality_params = {
    "historical_avg_temp_f": float(temp_mean_f),
    "temp_std_f": float(temp_std_f),
    "pct_change_per_degF": float(pct_change_per_degF),
    "mean_daily_deaths": float(model_df['death_count'].mean()),
    "dispersion_ratio": float(model_df['death_count'].var() / model_df['death_count'].mean()),
}
print(bloomington_mortality_params)

{'historical_avg_temp_f': 55.98124604012671, 'temp_std_f': 18.598988598342146, 'pct_change_per_degF': 0.02060998657593327, 'mean_daily_deaths': 0.26293558606124606, 'dispersion_ratio': 1.1398745086052509}
